# One scalar graph: PyTorch, then autograd from scratch

We will use one graph throughout:

$$
m=wx,\qquad a=m+b,\qquad e=a-y,\qquad L=e^2
$$

with $w=2$, $x=3$, $b=1$, and $y=10$.

First we let PyTorch differentiate it. Then we build the smallest useful version of the same idea
ourselves. The point is not to replace PyTorch—it is to see what `.backward()` does.

## 1 · The whole example in PyTorch

Each number below is a scalar tensor. We set `requires_grad=True` because we want PyTorch to calculate
its loss derivative. In ordinary training, the target `y` would normally be fixed; here we track it only
so the output matches our complete paper calculation.

PyTorch keeps `.grad` automatically for the four leaf tensors. `retain_grad()` asks it to keep gradients
for the intermediate values too.

In [1]:
import torch

torch.set_default_dtype(torch.float64)

w = torch.tensor(2.0, requires_grad=True)
x = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
y = torch.tensor(10.0, requires_grad=True)

# Forward pass
m = w * x
a = m + b
e = a - y
L = e ** 2

for node in (m, a, e, L):
    node.retain_grad()

print("Forward: m =", m.item(), ", a =", a.item(),
      ", e =", e.item(), ", L =", L.item())

Forward: m = 6.0 , a = 7.0 , e = -3.0 , L = 9.0


Now the important line:

In [2]:
L.backward()

print("w.grad =", w.grad)
print("x.grad =", x.grad)
print("b.grad =", b.grad)
print("y.grad =", y.grad)

w.grad = tensor(-18.)
x.grad = tensor(-12.)
b.grad = tensor(-6.)
y.grad = tensor(6.)


That is PyTorch autograd. The forward pass created a graph; `.backward()` sent a seed gradient of $1$
from $L$ through that graph in reverse.

For comparison with the paper calculation, here is every stored value:

In [3]:
torch_nodes = {"w": w, "x": x, "m": m, "b": b,
               "a": a, "y": y, "e": e, "L": L}
torch_reference = {
    name: (node.item(), node.grad.item())
    for name, node in torch_nodes.items()
}

print(f"{'node':<5} {'value':>8} {'grad':>8}")
for name, (value, grad) in torch_reference.items():
    print(f"{name:<5} {value:8.1f} {grad:8.1f}")

expected = {
    "w": (2, -18), "x": (3, -12), "m": (6, -6), "b": (1, -6),
    "a": (7, -6), "y": (10, 6), "e": (-3, -6), "L": (9, 1),
}
assert torch_reference == expected

node     value     grad
w          2.0    -18.0
x          3.0    -12.0
m          6.0     -6.0
b          1.0     -6.0
a          7.0     -6.0
y         10.0      6.0
e         -3.0     -6.0
L          9.0      1.0


## 2 · A tiny autograd engine from scratch

A `Value` stores only:

- its number in `data`,
- its accumulated loss-gradient buffer in `grad`, and
- ordered links to the operands that directly produced it.

Each parent link stores two things: the parent operand and the evaluated local derivative along that
edge. For example, $m=wx$ remembers $(w,\partial m/\partial w=x)$ and
$(x,\partial m/\partial x=w)$.

The gradient names are always relative to the operation currently running:

- <span style="color:#2C7A7B;font-weight:700">upstream</span>:
  $g_v=\partial L/\partial v$, already accumulated in `v.grad`;
- <span style="color:#2B6CB0;font-weight:700">local</span>:
  $\partial v/\partial u$, stored in the link from output $v$ to parent $u$;
- <span style="color:#EB811B;font-weight:700">downstream contribution</span>:
  $\Delta g_u=g_v(\partial v/\partial u)$, computed during backward and added to `u.grad`.

A parent is simply a direct input to an operation—not a whole neural-network layer.

In [4]:
class ParentLink:
    def __init__(self, value, local_grad):
        self.value = value
        self.local_grad = float(local_grad)

class Value:
    def __init__(self, data, label="", parents=(), op=""):
        self.data = float(data)
        self.grad = 0.0
        self.label = label
        self.parents = tuple(parents)
        self.op = op

def multiply(u, v, label):
    return Value(u.data * v.data, label,
                 parents=(ParentLink(u, v.data),
                          ParentLink(v, u.data)), op="×")

def add(u, v, label):
    return Value(u.data + v.data, label,
                 parents=(ParentLink(u, 1.0),
                          ParentLink(v, 1.0)), op="+")

def subtract(u, v, label):
    return Value(u.data - v.data, label,
                 parents=(ParentLink(u, 1.0),
                          ParentLink(v, -1.0)), op="−")

def square(u, label):
    return Value(u.data ** 2, label,
                 parents=(ParentLink(u, 2 * u.data),), op="²")

def backward(root):
    order, seen = [], set()

    # 1. Put every reachable value in forward order.
    def visit(node):
        if id(node) in seen:
            return
        seen.add(id(node))
        for link in node.parents:
            visit(link.value)
        order.append(node)

    visit(root)

    # 2. Clear old accumulated gradients, then seed the loss.
    for node in order:
        node.grad = 0.0
    root.grad = 1.0

    # 3. Walk backward. One parent link gives one chain-rule update.
    steps = []
    for node in reversed(order):
        for link in node.parents:
            upstream = node.grad
            local = link.local_grad
            downstream = upstream * local
            before = link.value.grad
            link.value.grad += downstream

            steps.append({
                "output": node.label,
                "upstream": upstream,
                "parent": link.value.label,
                "local": local,
                "downstream": downstream,
                "before": before,
                "after": link.value.grad,
            })
    return steps

Read `backward(root)` in three pieces:

1. `visit` follows the parent links and makes a forward order. Here it is
   `w, x, m, b, a, y, e, L`.
2. We clear the reachable `.grad` buffers and seed `L.grad = 1`, because
   $\partial L/\partial L=1$.
3. We reverse that order. At each link from output $v$ to parent $u$, the loop reads the
   upstream gradient from `v.grad`, reads the local derivative from the link, and adds their product
   into `u.grad`.

| quantity | where it lives |
|---|---|
| upstream $g_v$ | already accumulated in `v.grad` |
| local $\partial v/\partial u$ | saved in `link.local_grad` during the forward pass |
| downstream contribution $\Delta g_u$ | temporary variable `downstream` for this one edge |
| accumulated $g_u$ | updated in `link.value.grad` |

So the code does **not** store a separate downstream gradient forever. It computes one contribution,
adds it to the parent's buffer, and that buffer later becomes the upstream gradient for the parent.

In [5]:
#| echo: false
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("graphviz") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "graphviz"],
        check=True,
    )

from graphviz import Digraph
from IPython.display import HTML, display

def draw_graph(root, show_grad=True):
    nodes, seen = [], set()
    def visit(node):
        if id(node) in seen:
            return
        seen.add(id(node))
        nodes.append(node)
        for link in node.parents:
            visit(link.value)
    visit(root)

    dot = Digraph(format="svg")
    dot.attr(rankdir="LR", bgcolor="transparent", pad="0.15",
             nodesep="0.25", ranksep="0.45")
    dot.attr("node", fontname="Helvetica", fontsize="11")
    dot.attr("edge", fontname="Helvetica", fontsize="9", color="#60777b")

    for node in nodes:
        node_id = "value_" + node.label
        grad = f"{node.grad:g}" if show_grad else "—"
        fill = "#e8f7f5" if show_grad and node.grad != 0 else "#ffffff"
        border = "#2C7A7B" if show_grad and node.grad != 0 else "#1f3a40"
        dot.node(
            node_id,
            label=f"{{ {node.label} | value {node.data:g} | grad {grad} }}",
            shape="record", style="rounded,filled", fillcolor=fill,
            color=border, fontcolor="#1f3a40",
        )
        if node.op:
            op_id = "op_" + node.label
            dot.node(op_id, label=node.op, shape="circle", fixedsize="true",
                     width="0.32", style="filled", fillcolor="#2B6CB0",
                     color="#2B6CB0", fontcolor="white")
            dot.edge(op_id, node_id)
            for link in node.parents:
                local_label = (
                    f"∂{node.label}/∂{link.value.label}="
                    f"{link.local_grad:g}"
                )
                dot.edge(
                    "value_" + link.value.label,
                    op_id,
                    label=local_label,
                    fontcolor="#2B6CB0",
                )
    svg = dot.pipe(format="svg").decode("utf-8")
    svg = svg.replace(
        "<svg ",
        '<svg style="width:100%;height:auto;min-width:780px" ',
        1,
    )
    return HTML(
        '<div style="max-width:100%;overflow-x:auto">'
        '<div style="min-width:780px">' + svg + '</div></div>'
    )

def show_backward_steps(steps):
    rows = []
    for number, step in enumerate(steps, start=1):
        output = step["output"]
        parent = step["parent"]
        rows.append(
            "<div style='border:1px solid #d6e2e1;border-radius:8px;"
            "padding:10px 12px;margin:8px 0;background:#fff'>"
            f"<div style='font-weight:700;margin-bottom:4px'>{number}. {output} → {parent}</div>"
            "<div style='font-size:1.02em;line-height:1.55'>"
            f"<span style='color:#2C7A7B;font-weight:700'>g<sub>{output}</sub> = {step['upstream']:g}</span>"
            " &nbsp;×&nbsp; "
            f"<span style='color:#2B6CB0;font-weight:700'>∂{output}/∂{parent} = {step['local']:g}</span>"
            " &nbsp;=&nbsp; "
            f"<span style='color:#EB811B;font-weight:700'>Δg<sub>{parent}</sub> = {step['downstream']:g}</span>"
            "</div>"
            f"<div style='color:#526669;margin-top:3px'>{parent}.grad: "
            f"{step['before']:g} → {step['after']:g}</div></div>"
        )

    display(HTML(
        "<div style='border-left:5px solid #2C7A7B;background:#eef8f7;"
        "padding:10px 12px;margin:4px 0 12px;border-radius:4px'>"
        "<b>Seed:</b> L.grad = ∂L/∂L = 1</div>" + "".join(rows)
    ))

Build the **same forward graph**, one readable line per operation. On a phone, scroll the graph sideways:

In [6]:
sw = Value(2.0, label="w")
sx = Value(3.0, label="x")
sb = Value(1.0, label="b")
sy = Value(10.0, label="y")

sm = multiply(sw, sx, "m")
sa = add(sm, sb, "a")
se = subtract(sa, sy, "e")
sL = square(se, "L")

print("What m=wx stored during forward:")
for link in sm.parents:
    print(f"  parent {link.value.label}: local ∂m/∂{link.value.label} = {link.local_grad:g}")

draw_graph(sL, show_grad=False)

What m=wx stored during forward:
  parent w: local ∂m/∂w = 3
  parent x: local ∂m/∂x = 2


Now our whole reverse pass is also one line:

In [7]:
steps = backward(sL)
show_backward_steps(steps)

draw_graph(sL, show_grad=True)

The trace contains every reverse edge. For example, the square sends $-6$ into `e.grad`. On the next
operation, that same stored number becomes the upstream gradient $g_e$ for subtraction.

A single row's product is a **downstream contribution**. If several paths return to one value, each row
adds into the same `parent.grad` buffer; only their sum is the full gradient at that parent.

Finally, check that our tiny engine and PyTorch agree at every named value.

In [8]:
scratch_nodes = {"w": sw, "x": sx, "m": sm, "b": sb,
                 "a": sa, "y": sy, "e": se, "L": sL}

for name, node in scratch_nodes.items():
    torch_value, torch_grad = torch_reference[name]
    assert node.data == torch_value
    assert node.grad == torch_grad

print("✓ Every value and gradient matches PyTorch.")

✓ Every value and gradient matches PyTorch.


## Takeaway

Both systems do the same three things:

1. run the forward operations and store parent links plus local derivatives,
2. start with $g_L=1$ in the loss's `.grad` buffer,
3. compute <span style="color:#2C7A7B;font-weight:700">upstream</span>
   $\times$ <span style="color:#2B6CB0;font-weight:700">local</span>
   $=$ <span style="color:#EB811B;font-weight:700">downstream contribution</span>, then add it to
   the parent's `.grad` buffer.

Our tiny `Value` record and four local rules make those steps visible. PyTorch generalizes them to
tensors, neural-network layers, accelerators, and large models.